# Human-in-the-Loop Approval with LangGraph
## Pause the Graph — Human Approval Before Action
⏱ ~10 min

**Human-in-the-Loop (HITL)** is the pattern for keeping a human approval step inside an otherwise autonomous agent pipeline. LangGraph implements it through two primitives: `interrupt()` to pause the graph mid-run and surface a decision point, and `Command(resume=...)` to hand the human's decision back in and continue.

By the end of this workshop you will understand *why* HITL exists, *how* the interrupt/resume mechanism works under the hood, and *how* to wire it into your own LangGraph agents.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why HITL? Fully autonomous vs human-gated patterns |
| 2 | **Core Primitives** — `interrupt()`, `MemorySaver`, `Command(resume=...)` |
| 3 | **Building the Graph** — State, nodes, edges, checkpointer |
| 4 | **Running the Interrupt** — First `.stream()` call, reading the pause event |
| 5 | **Resuming** — Second `.stream()` call with `Command`, approve and reject paths |
| 6 | **Inspecting Saved State** — `app.get_state()` and checkpoint introspection |
| 7 | **Multi-Step Approval** — Chaining multiple interrupts in one graph |
| ★ | **Advanced** — Structured approval payloads, persistent checkpointers |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- Familiarity with LangGraph basics (see example 5-react-agent-lg)

### Key References
> Amershi, S., Weld, D., et al. (2019). *Software Engineering for Machine Learning: A Case Study.* ICSE 2019. https://doi.org/10.1109/ICSE-SEIP.2019.00042
>
> Shneiderman, B. (2020). *Human-Centered AI: Reliable, Safe & Trustworthy.* https://doi.org/10.1162/isala_a_00001
>
> LangGraph interrupt docs: https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/
>
> LangGraph `Command` reference: https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Command

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid.\n"
    "  Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: {bool(key)}")

---

## Part 1 — What Is HITL and Why Does It Exist?

---

### The Problem

Autonomous agents can take **irreversible actions**: deleting records, sending emails, deploying code, charging customers. When an agent is wrong — due to a misunderstood instruction, a hallucinated fact, or a context drift — those actions cannot be undone.

Human-in-the-Loop (HITL) inserts a **mandatory human checkpoint** before irreversible steps. The agent drafts the action; a human reviews and approves or rejects it; only then does the action execute.

---

### Three Patterns Compared

| Pattern | Human role | Latency | Risk | Best for |
|---------|-----------|---------|------|----------|
| **Fully autonomous** | None — agent acts immediately | Lowest | High for irreversible ops | Low-stakes, reversible tasks |
| **Human-in-the-loop** | Reviews and approves before each key action | Medium | Low | Destructive or high-value actions |
| **Human-on-the-loop** | Gets notified after the fact; can override | Low | Medium | Monitoring, audit trails |

HITL is the right choice whenever the cost of a wrong action exceeds the cost of a short wait for a human review.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **interrupt()** | LangGraph primitive that pauses a graph node mid-execution and surfaces a value to the caller |
| **MemorySaver** | In-memory checkpointer that persists graph state across the pause |
| **thread_id** | Unique identifier that ties the pause and resume calls to the same saved checkpoint |
| **Command(resume=...)** | Object passed to the second `.stream()` call to inject the human decision and continue |
| **checkpoint** | Snapshot of the entire graph state saved by MemorySaver at the interrupt point |
| **`__interrupt__`** | Special key in the streaming chunk dict that signals the graph is paused |
| **re-run semantics** | On resume, LangGraph re-executes the interrupted node from its top — code before `interrupt()` runs twice |

### HITL Architecture

```
FIRST .stream() CALL — runs until interrupt() fires
────────────────────────────────────────────────────────

  app.stream(initial_state, config)
       │
       ▼
  ┌─────────────┐
  │ draft_action│  Agent prepares the proposed action
  └──────┬──────┘  and writes it to graph state
         │
         ▼
  ┌──────────────────┐
  │  await_approval  │
  │                  │
  │  interrupt(value)│──── PAUSES HERE ────────────────┐
  │                  │  state saved to MemorySaver      │
  └──────────────────┘  chunk["__interrupt__"] emitted  │
                                                         │
  Caller reads the interrupt value:                      │
  { "action": "...", "question": "Approve?" }            │
                                                         │
                                                         │
HUMAN DECISION                                           │
────────────────────────────────────────────────────────  │
                                                         │
  Human reads the proposed action                        │
  Human clicks Approve or Reject                         │
                                                         │
                                                         │
SECOND .stream() CALL — resumes from checkpoint ◄────────┘
────────────────────────────────────────────────────────

  app.stream(Command(resume=True/False), config)
       │
       ▼
  ┌──────────────────┐
  │  await_approval  │  Node re-runs from TOP
  │                  │  interrupt() returns the decision
  │  decision = True │
  └──────┬─────┬─────┘
         │     │
    True ▼     ▼ False
  EXECUTED   CANCELLED
       │           │
       └─────┬─────┘
             ▼
            END
```

> **Key insight:** `interrupt()` works like `yield` in a generator — it suspends execution and returns a value to the caller. The MemorySaver checkpointer saves the entire graph state so that when `Command(resume=...)` arrives, LangGraph can restore the node's local variables and continue exactly where it stopped.

---

## Part 2 — Core Primitives

---

### `interrupt(value)` — The Pause Gate

```python
from langgraph.types import interrupt

def await_approval(state):
    # Code here runs TWICE (initial run + resume)
    decision = interrupt({"action": state["action"], "question": "Approve?"})
    # Code here runs ONCE (only after resume, when interrupt() returns)
    if decision:
        return {"result": "EXECUTED"}
    return {"result": "CANCELLED"}
```

The `value` you pass to `interrupt()` is what appears in the `__interrupt__` streaming chunk. It can be any JSON-serializable object — a string, a dict, a list.

---

### `MemorySaver` — The Checkpointer

```python
from langgraph.checkpoint.memory import MemorySaver

app = graph.compile(checkpointer=MemorySaver())
#                   ^^^^^^^^^^^^^^^^^^^^^^^^^
#  Without this, resume raises: RuntimeError: No checkpointer set
```

MemorySaver stores snapshots in-process RAM. For production, swap it for a persistent checkpointer (SQLite, Redis, Postgres).

---

### `thread_id` — The Conversation Key

```python
config = {"configurable": {"thread_id": "my-unique-id"}}
#                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#  Same thread_id in BOTH .stream() calls links pause → resume
#  Different thread_id = fresh run with no saved state
```

---

### `Command(resume=...)` — The Resume Signal

```python
from langgraph.types import Command

# Resume with approval
app.stream(Command(resume=True), config)

# Resume with rejection
app.stream(Command(resume=False), config)

# Resume with structured data (the dict becomes interrupt()'s return value)
app.stream(Command(resume={"approved": True, "reviewer": "alice"}), config)
```

---

### Primitive Comparison Table

| Primitive | Where used | Purpose | Required? |
|-----------|-----------|---------|----------|
| `interrupt(value)` | Inside a node function | Pause the graph and emit value to caller | Yes |
| `MemorySaver` | `graph.compile(checkpointer=...)` | Save state across the pause | Yes |
| `thread_id` | `config["configurable"]` | Link pause and resume calls | Yes |
| `Command(resume=...)` | Second `.stream()` argument | Inject human decision and continue | Yes |
| `stream_mode="updates"` | `.stream()` keyword | Emit per-node dicts instead of full state | Recommended |

---

## Part 3 — Building the HITL Graph

---

### Graph Structure

```
START
  ↓
draft_action    ← writes proposed action to state
  ↓
await_approval  ← interrupt() pauses here
  ↓
END
```

Two nodes, one interrupt point. Simple enough to see every primitive clearly.

In [ ]:
# ─── 3-A: Define graph state ──────────────────────────────────────────────────
# TypedDict is the standard state container for LangGraph.
# Every key is readable and writable by any node.

from typing import Optional, TypedDict


class HITLState(TypedDict):
    action: str     # the proposed action drafted by the agent
    approved: bool  # human's decision (set after resume)
    result: str     # final outcome message


# Inspect what an empty initial state looks like
initial_state: HITLState = {"action": "", "approved": False, "result": ""}
print("Initial state:", initial_state)
print("State keys:", list(HITLState.__annotations__.keys()))

In [ ]:
# ─── 3-B: Define the two nodes ────────────────────────────────────────────────
from langgraph.types import interrupt

PENDING_ACTION = "Delete all records older than 90 days from the analytics database"


def draft_action(state: HITLState) -> dict:
    """Node 1: prepare the action the agent wants to take.

    In a real agent this would call an LLM to decide what action
    to propose. Here we hardcode it for clarity.
    """
    print(f"[draft_action] Proposing: {PENDING_ACTION}")
    return {"action": PENDING_ACTION}


def await_approval(state: HITLState) -> dict:
    """Node 2: pause and wait for human approval.

    IMPORTANT — re-run semantics:
      On resume, LangGraph re-executes this node from the very top.
      Any code BEFORE interrupt() therefore runs twice:
        - Run 1: graph reaches interrupt(), pauses, emits the value
        - Run 2: resume arrives, node re-starts, interrupt() returns the decision

    Rule: put all side effects (writes, emails, API calls) AFTER interrupt().
    """
    print(f"[await_approval] Requesting approval for: {state['action']}")

    # interrupt() pauses here on Run 1. On Run 2 it returns the human's decision.
    decision = interrupt({"action": state["action"], "question": "Approve this action?"})

    # Side effects go here (only reached after resume)
    if decision:
        print("[await_approval] Decision: APPROVED")
        return {"approved": True, "result": f"EXECUTED: {state['action']}"}
    print("[await_approval] Decision: REJECTED")
    return {"approved": False, "result": f"CANCELLED: {state['action']}"}


print("Nodes defined: draft_action, await_approval")

In [ ]:
# ─── 3-C: Assemble and compile the graph ──────────────────────────────────────
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

graph = StateGraph(HITLState)

# Add nodes
graph.add_node("draft_action", draft_action)
graph.add_node("await_approval", await_approval)

# Wire the edges
graph.add_edge(START, "draft_action")
graph.add_edge("draft_action", "await_approval")
graph.add_edge("await_approval", END)

# Compile with MemorySaver — mandatory for interrupt/resume
# Without a checkpointer: RuntimeError: No checkpointer set
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Graph compiled successfully.")
print("Checkpointer:", type(memory).__name__)
print("Nodes:", list(graph.nodes))

---

## Part 4 — Running the Interrupt

---

### `.stream()` vs `.invoke()` for HITL

| Method | Behavior at interrupt | Use for |
|--------|-----------------------|---------|
| `.invoke()` | **Raises** `GraphInterrupt` exception | Not suitable for HITL notebooks |
| `.stream()` | **Yields** `{"__interrupt__": [...]}` chunk then stops | The correct approach — lets you catch the pause |

When streaming with `stream_mode="updates"`, each chunk is a dict of `{node_name: state_updates}`. The interrupt chunk has the special key `"__interrupt__"` instead of a node name.

### Reading the Interrupt Chunk

```python
# The streaming chunk at interrupt time looks like:
{
    "__interrupt__": [
        Interrupt(
            value={"action": "Delete all records...", "question": "Approve?"},
            resumable=True,
            ns=["await_approval:..."]
        )
    ]
}

# Access the value you passed to interrupt():
interrupt_value = chunk["__interrupt__"][0].value
```

In [ ]:
# ─── 4-A: First .stream() — runs until interrupt() fires ──────────────────────
# The thread_id ties this call to any future resume call.
# Think of it as a conversation session ID.

config = {"configurable": {"thread_id": "hitl-demo-001"}}

print("Starting first .stream() call...")
print("─" * 50)

interrupted_value = None

for chunk in app.stream(
    {"action": "", "approved": False, "result": ""},
    config,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        # Graph is paused — extract the value passed to interrupt()
        interrupted_value = chunk["__interrupt__"][0].value
        print(f"\n*** GRAPH PAUSED ***")
        print(f"Interrupt value: {interrupted_value}")
        break
    for node, update in chunk.items():
        print(f"Chunk from [{node}]: {update}")

print("─" * 50)
if interrupted_value:
    print("\nGraph is now suspended. State saved to MemorySaver.")
    print(f"  Proposed action : {interrupted_value['action']}")
    print(f"  Question        : {interrupted_value['question']}")

In [ ]:
# ─── 4-B: Inspect the saved checkpoint state ──────────────────────────────────
# app.get_state() returns the full snapshot saved by MemorySaver.
# Use this to confirm the graph is suspended and read the current state values.

snapshot = app.get_state(config)

print("Checkpoint snapshot:")
print(f"  values    : {snapshot.values}")
print(f"  next nodes: {snapshot.next}")
print(f"  tasks     : {snapshot.tasks}")
print()
print("Observation: next=['await_approval'] means the graph is paused inside that node.")
print("'result' is still empty — nothing has been written yet (interrupt fires before the return).")

---

## Part 5 — Resuming with a Human Decision

---

### How Resume Works

When you call `app.stream(Command(resume=value), config)`, LangGraph:

1. Loads the checkpoint saved under `thread_id`
2. Re-executes `await_approval` from the **top** of the function
3. When execution reaches `interrupt()` again, instead of pausing, it **returns `value`** immediately
4. The rest of the node runs normally and the graph continues to END

This is why code before `interrupt()` runs twice — it's a full node re-execution, not a continuation mid-function.

In [ ]:
# ─── 5-A: Resume with APPROVAL ────────────────────────────────────────────────
# Command(resume=True) injects True as the return value of interrupt().
# The await_approval node picks up after interrupt() and sees decision=True.

from langgraph.types import Command

print("Resuming with: APPROVED")
print("─" * 50)

for chunk in app.stream(Command(resume=True), config, stream_mode="updates"):
    if "__interrupt__" in chunk:
        print("(another interrupt — not expected in this path)")
        break
    for node, update in chunk.items():
        print(f"Chunk from [{node}]: {update}")

print("─" * 50)

# Read final state
final_state = app.get_state(config).values
print(f"\nFinal result: {final_state['result']}")
print(f"Approved    : {final_state['approved']}")

In [ ]:
# ─── 5-B: Full run with REJECTION — fresh thread ──────────────────────────────
# Using a new thread_id starts a clean run with no saved state.
# This demonstrates the cancellation path.

config_reject = {"configurable": {"thread_id": "hitl-demo-002"}}

print("=== REJECTION PATH ===")
print("Step 1: First .stream() — runs until interrupt")
print("─" * 50)

for chunk in app.stream(
    {"action": "", "approved": False, "result": ""},
    config_reject,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        iv = chunk["__interrupt__"][0].value
        print(f"PAUSED — action: {iv['action']}")
        break
    for node, update in chunk.items():
        print(f"[{node}]: {update}")

print()
print("Step 2: Resume with rejection")
print("─" * 50)

for chunk in app.stream(Command(resume=False), config_reject, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"[{node}]: {update}")

final_reject = app.get_state(config_reject).values
print(f"\nFinal result: {final_reject['result']}")
print(f"Approved    : {final_reject['approved']}")

In [ ]:
# ─── 5-C: Side-by-side comparison of approve vs reject outcomes ───────────────

approved_result = app.get_state(config).values
rejected_result = app.get_state(config_reject).values

print("=" * 60)
print(f"{'PATH':<12} {'approved':<10} {'result'}")
print("=" * 60)
print(f"{'APPROVED':<12} {str(approved_result['approved']):<10} {approved_result['result']}")
print(f"{'REJECTED':<12} {str(rejected_result['approved']):<10} {rejected_result['result']}")
print("=" * 60)

---

## Part 6 — Inspecting Saved State

---

`app.get_state(config)` is the primary debugging tool for HITL graphs. It returns a `StateSnapshot` with:

| Attribute | Type | What it shows |
|-----------|------|---------------|
| `.values` | dict | Current state values for all keys |
| `.next` | tuple | Node(s) that will run next on resume (empty = graph is at END) |
| `.tasks` | tuple | Pending tasks (contains interrupt details when paused) |
| `.config` | dict | The config including thread_id |
| `.metadata` | dict | Step count, source, write count |

You can also use `app.get_state_history(config)` to see every checkpoint, step by step.

In [ ]:
# ─── 6-A: Walk the full state history of a completed run ──────────────────────
# get_state_history() yields snapshots from newest to oldest.

print("State history for APPROVED run (thread: hitl-demo-001):")
print("─" * 60)

for i, snap in enumerate(app.get_state_history(config)):
    print(f"\nStep {i}:")
    print(f"  next   : {snap.next}")
    print(f"  values : {snap.values}")
    print(f"  metadata step: {snap.metadata.get('step', '?')}")

In [ ]:
# ─── 6-B: Capture the mid-pause snapshot on a fresh run ───────────────────────
# Start a new run but DON'T resume — examine what MemorySaver holds at the pause.

config_inspect = {"configurable": {"thread_id": "hitl-inspect-001"}}

# Run until paused
for chunk in app.stream(
    {"action": "", "approved": False, "result": ""},
    config_inspect,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        break

# Inspect the paused state
paused_snap = app.get_state(config_inspect)

print("Snapshot at interrupt point:")
print(f"  values : {paused_snap.values}")
print(f"  next   : {paused_snap.next}   <- still in await_approval")
print(f"  tasks  : {paused_snap.tasks}")
print()
print("Key observations:")
print("  1. 'action' IS populated — draft_action ran to completion.")
print("  2. 'result' IS EMPTY — the return statement in await_approval hasn't run yet.")
print("  3. 'next' contains 'await_approval' — resume will re-enter this node.")

---

## Part 7 — Multi-Step Approval: Chaining Interrupts

---

Real pipelines often need **multiple approval gates** — for example:
1. Manager approves the plan
2. Security team approves the scope
3. System executes

LangGraph supports multiple `interrupt()` calls in the same graph (even in the same node). Each one generates a separate pause-resume cycle on the same `thread_id`.

```
START → plan_action → manager_gate ─(interrupt)─► [pause 1]
                                                       │
                                               human resumes
                                                       │
                    security_gate ─(interrupt)─► [pause 2]
                                                       │
                                               human resumes
                                                       │
                    execute → END
```

In [ ]:
# ─── 7-A: Graph with two sequential interrupt points ──────────────────────────


class MultiApprovalState(TypedDict):
    action: str
    manager_approved: Optional[bool]
    security_approved: Optional[bool]
    result: str


def plan_action(state: MultiApprovalState) -> dict:
    return {"action": "Migrate 500 GB of user data to new data centre"}


def manager_gate(state: MultiApprovalState) -> dict:
    decision = interrupt({
        "gate": "MANAGER REVIEW",
        "action": state["action"],
        "question": "Manager: approve this migration?",
    })
    if not decision:
        return {"manager_approved": False, "result": "BLOCKED by manager"}
    return {"manager_approved": True}


def security_gate(state: MultiApprovalState) -> dict:
    decision = interrupt({
        "gate": "SECURITY REVIEW",
        "action": state["action"],
        "question": "Security: approve data transfer?",
    })
    if not decision:
        return {"security_approved": False, "result": "BLOCKED by security"}
    return {"security_approved": True}


def execute_action(state: MultiApprovalState) -> dict:
    return {"result": f"EXECUTED: {state['action']}"}


def route_after_manager(state: MultiApprovalState) -> str:
    """Conditional edge: skip to END if manager rejected."""
    return "security_gate" if state["manager_approved"] else END


def route_after_security(state: MultiApprovalState) -> str:
    """Conditional edge: skip to END if security rejected."""
    return "execute_action" if state["security_approved"] else END


multi_graph = StateGraph(MultiApprovalState)
multi_graph.add_node("plan_action", plan_action)
multi_graph.add_node("manager_gate", manager_gate)
multi_graph.add_node("security_gate", security_gate)
multi_graph.add_node("execute_action", execute_action)

multi_graph.add_edge(START, "plan_action")
multi_graph.add_edge("plan_action", "manager_gate")
multi_graph.add_conditional_edges("manager_gate", route_after_manager)
multi_graph.add_conditional_edges("security_gate", route_after_security)
multi_graph.add_edge("execute_action", END)

multi_app = multi_graph.compile(checkpointer=MemorySaver())
print("Multi-interrupt graph compiled.")

In [ ]:
# ─── 7-B: Run the two-gate approval flow ──────────────────────────────────────


def run_until_interrupt(graph_app, cfg, initial=None):
    """Helper: stream until __interrupt__ or END, return interrupt value or None."""
    input_data = initial if initial is not None else Command(resume=True)
    for chunk in graph_app.stream(input_data, cfg, stream_mode="updates"):
        if "__interrupt__" in chunk:
            iv = chunk["__interrupt__"][0].value
            print(f"\n[PAUSED at {iv['gate']}]")
            print(f"  Action   : {iv['action']}")
            print(f"  Question : {iv['question']}")
            return iv
        for node, update in chunk.items():
            print(f"  [{node}]: {update}")
    return None  # graph reached END


cfg_multi = {"configurable": {"thread_id": "multi-gate-001"}}
initial_state_multi = {"action": "", "manager_approved": None, "security_approved": None, "result": ""}

print("=== TWO-GATE APPROVAL RUN (both gates: APPROVE) ===")
print()

# Gate 1: manager
print("Gate 1 — Manager review:")
iv1 = run_until_interrupt(multi_app, cfg_multi, initial_state_multi)

if iv1:
    print("  Decision: APPROVED")
    # Gate 2: security (resume=True means previous gate approved)
    print("\nGate 2 — Security review:")
    iv2 = run_until_interrupt(multi_app, cfg_multi)

    if iv2:
        print("  Decision: APPROVED")
        # Execute
        print("\nFinal execution:")
        run_until_interrupt(multi_app, cfg_multi)

final = multi_app.get_state(cfg_multi).values
print(f"\nFinal result  : {final['result']}")
print(f"Manager OK    : {final['manager_approved']}")
print(f"Security OK   : {final['security_approved']}")

---

## Exercises

---

### Exercise 1 — Reject the Action

Start a fresh run with a new `thread_id` (e.g., `"hitl-ex1"`) and resume with `Command(resume=False)`. What `result` string does the graph produce? Confirm with `app.get_state(config).values`.

### Exercise 2 — Add a Reviewer Field to the State

Modify `HITLState` to include `reviewer: str`. Change `Command(resume=...)` to pass a dict like `{"approved": True, "reviewer": "alice"}`. Update `await_approval` to extract both fields from the return value of `interrupt()` and include the reviewer name in the `result` string.

### Exercise 3 — Block Manager Gate

In the multi-gate graph from Part 7, resume Gate 1 with `Command(resume=False)`. Verify that the security gate is never reached and the result shows `"BLOCKED by manager"`.

### Exercise 4 — Persistent Checkpointer

Replace `MemorySaver()` with `SqliteSaver.from_conn_string(":memory:")` from `langgraph.checkpoint.sqlite`. Confirm the interrupt/resume flow still works. (Hint: `pip install langgraph-checkpoint-sqlite`)

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Start a fresh run and reject the action. Inspect the final state.

config_ex1 = {"configurable": {"thread_id": "hitl-ex1"}}

# TODO: run app.stream() with the initial state until __interrupt__
# TODO: resume with Command(resume=False)
# TODO: print app.get_state(config_ex1).values and observe 'result' and 'approved'

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Add a reviewer field. Resume with a dict payload.

class HITLStateV2(TypedDict):
    action: str
    approved: bool
    reviewer: str  # NEW
    result: str


def await_approval_v2(state: HITLStateV2) -> dict:
    # TODO: pass a richer interrupt value, and extract 'approved' and 'reviewer'
    # from the dict that Command(resume={...}) injects
    decision = interrupt({"action": state["action"], "question": "Approve this action?"})
    # TODO: handle dict decision: decision["approved"], decision["reviewer"]
    pass


# TODO: rebuild the graph with HITLStateV2 and await_approval_v2
# TODO: resume with Command(resume={"approved": True, "reviewer": "alice"})
# TODO: confirm 'result' includes the reviewer name

---

## Part 8 ★ — Advanced Techniques (Bonus)

---

### Structured Approval Payloads

Instead of a bare `True/False`, pass a full context dict to `interrupt()` so the human reviewer has everything they need:

```python
decision = interrupt({
    "action": state["action"],
    "risk_level": "HIGH",
    "estimated_impact": "~50,000 rows deleted",
    "reversible": False,
    "question": "Type 'CONFIRM' to approve or 'CANCEL' to reject",
})
```

### Timeout Simulation

MemorySaver holds state indefinitely. In production you would wrap the resume call with a deadline check and auto-reject after N minutes:

```python
import time
APPROVAL_TIMEOUT_SECONDS = 300  # 5 minutes

def try_resume(app, config, decision, deadline):
    if time.time() > deadline:
        # Auto-reject on timeout
        return app.stream(Command(resume=False), config, stream_mode="updates")
    return app.stream(Command(resume=decision), config, stream_mode="updates")
```

### LLM-as-Judge Approval

For lower-stakes actions, replace the human with an LLM judge (see example 29-llm-judge-harness):

```python
def auto_review_gate(state):
    # Pause and surface the action to the judge
    action_payload = interrupt({"action": state["action"]})
    # The judge calls Command(resume=judge_decision)
    ...
```

### Persistent Checkpointers for Production

| Checkpointer | Package | Best for |
|-------------|---------|----------|
| `MemorySaver` | `langgraph` (built-in) | Development and notebooks |
| `SqliteSaver` | `langgraph-checkpoint-sqlite` | Single-process production |
| `RedisSaver` | `langgraph-checkpoint-redis` | Multi-process / multi-server |
| `PostgresSaver` | `langgraph-checkpoint-postgres` | Enterprise / high availability |

In [ ]:
# ─── 8-A: Structured interrupt payload with dict resume ───────────────────────
# Demonstrate passing a rich context dict through interrupt().
# The resume value is also a dict with 'approved' and 'reviewer'.


class RichState(TypedDict):
    action: str
    risk_level: str
    approved: bool
    reviewer: str
    result: str


def draft_risky_action(state: RichState) -> dict:
    return {
        "action": "Purge all PII from the EU data warehouse",
        "risk_level": "HIGH",
    }


def rich_approval_gate(state: RichState) -> dict:
    # Surface a rich context payload to the reviewer
    decision = interrupt({
        "action": state["action"],
        "risk_level": state["risk_level"],
        "reversible": False,
        "estimated_rows": 2_400_000,
        "question": "GDPR purge request — approve?",
    })
    # decision is the dict we pass in Command(resume={...})
    approved = decision.get("approved", False) if isinstance(decision, dict) else bool(decision)
    reviewer = decision.get("reviewer", "unknown") if isinstance(decision, dict) else "notebook"

    if approved:
        return {
            "approved": True,
            "reviewer": reviewer,
            "result": f"EXECUTED by {reviewer}: {state['action']}",
        }
    return {
        "approved": False,
        "reviewer": reviewer,
        "result": f"CANCELLED by {reviewer}: {state['action']}",
    }


rich_graph = StateGraph(RichState)
rich_graph.add_node("draft", draft_risky_action)
rich_graph.add_node("gate", rich_approval_gate)
rich_graph.add_edge(START, "draft")
rich_graph.add_edge("draft", "gate")
rich_graph.add_edge("gate", END)
rich_app = rich_graph.compile(checkpointer=MemorySaver())

cfg_rich = {"configurable": {"thread_id": "rich-001"}}

# First stream — run until interrupt
for chunk in rich_app.stream(
    {"action": "", "risk_level": "", "approved": False, "reviewer": "", "result": ""},
    cfg_rich,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        iv = chunk["__interrupt__"][0].value
        print("PAUSED — Rich interrupt payload:")
        for k, v in iv.items():
            print(f"  {k:<18}: {v}")
        break

# Resume with a structured decision dict
print("\nResuming with: {approved: True, reviewer: 'dpo@company.com'}")
for chunk in rich_app.stream(
    Command(resume={"approved": True, "reviewer": "dpo@company.com"}),
    cfg_rich,
    stream_mode="updates",
):
    for node, update in chunk.items():
        if update.get("result"):
            print(f"\n[{node}] {update['result']}")

rich_final = rich_app.get_state(cfg_rich).values
print(f"Reviewer: {rich_final['reviewer']}")

---

## What's Next?

You now understand the full interrupt/resume lifecycle in LangGraph. Here's where to go deeper:

### Apply HITL to real agents
- **Example 5 — ReAct Agent with MemorySaver** (`../5-react-agent-lg/`): MemorySaver for multi-turn conversation memory — the same checkpointer used here. Add `interrupt()` before any tool call to gate it.
- **Example 30 — Agentic RAG** (`../30-agentic-rag/`): A retrieval agent that takes tool-calling actions. Wire an interrupt before each web search or database write.

### Replace the human with automation
- **Example 29 — LLM Judge Harness** (`../29-llm-judge-harness/`): Use an LLM as the reviewer. The judge calls `Command(resume=judge_decision)` instead of a human — same graph, different approver.

### Go to production
- **Persistent checkpointers**: Swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` so approvals survive server restarts.
- **Webhook integration**: In a web app, the first `.stream()` call writes the interrupt to a database row; a POST /approve endpoint calls the second `.stream()` with `Command(resume=decision)`.
- **LangSmith tracing**: Add `LANGCHAIN_TRACING_V2=true` to see the full interrupt timeline in the LangSmith dashboard.

### Further reading
- LangGraph HITL concepts: https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/
- LangGraph `interrupt()` API: https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.interrupt
- Amershi et al. (2019). *Software Engineering for Machine Learning.* ICSE 2019.
- Shneiderman (2020). *Human-Centered AI.* MIT Press.

---

*Workshop complete. The next step is example 30 — add HITL approval to a full agentic RAG pipeline.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Reject the Action

**Expected result:** `result = 'CANCELLED: Delete all records older than 90 days from the analytics database'` and `approved = False`.

The key point: the graph still runs to completion — the rejection path is a normal branch, not an error or exception. The `result` key gets written with the `CANCELLED:` prefix by the else branch in `await_approval`.

In [ ]:
# Answer — Exercise 1
config_ans1 = {"configurable": {"thread_id": "hitl-ans1"}}

# Step 1: run until interrupt
for chunk in app.stream(
    {"action": "", "approved": False, "result": ""},
    config_ans1,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        print("Paused. Resuming with rejection...")
        break

# Step 2: resume with False
for chunk in app.stream(Command(resume=False), config_ans1, stream_mode="updates"):
    for node, update in chunk.items():
        if update.get("result"):
            print(f"[{node}] {update['result']}")

ans1_state = app.get_state(config_ans1).values
print(f"approved : {ans1_state['approved']}")
print(f"result   : {ans1_state['result']}")

### Exercise 2 — Add a Reviewer Field

**Key design points:**
- `interrupt()` returns whatever value you pass to `Command(resume=...)` — so pass a dict, get a dict back.
- Guard against receiving a plain `bool` (if someone forgets to use the dict form) with `isinstance(decision, dict)`.
- The reviewer name in `result` makes the audit trail immediately readable.

In [ ]:
# Answer — Exercise 2

class HITLStateV2Answer(TypedDict):
    action: str
    approved: bool
    reviewer: str
    result: str


def await_approval_v2_answer(state: HITLStateV2Answer) -> dict:
    decision = interrupt({
        "action": state["action"],
        "question": "Approve? Respond with {approved: bool, reviewer: str}",
    })
    # Handle both dict and plain bool for flexibility
    if isinstance(decision, dict):
        approved = decision.get("approved", False)
        reviewer = decision.get("reviewer", "unknown")
    else:
        approved = bool(decision)
        reviewer = "notebook"

    verb = "EXECUTED" if approved else "CANCELLED"
    return {
        "approved": approved,
        "reviewer": reviewer,
        "result": f"{verb} by {reviewer}: {state['action']}",
    }


g2 = StateGraph(HITLStateV2Answer)
g2.add_node("draft_action", lambda s: {"action": PENDING_ACTION})
g2.add_node("await_approval", await_approval_v2_answer)
g2.add_edge(START, "draft_action")
g2.add_edge("draft_action", "await_approval")
g2.add_edge("await_approval", END)
app2 = g2.compile(checkpointer=MemorySaver())

cfg2 = {"configurable": {"thread_id": "hitl-ans2"}}

for chunk in app2.stream(
    {"action": "", "approved": False, "reviewer": "", "result": ""},
    cfg2,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        break

for chunk in app2.stream(
    Command(resume={"approved": True, "reviewer": "alice@company.com"}),
    cfg2,
    stream_mode="updates",
):
    for node, update in chunk.items():
        if update.get("result"):
            print(f"[{node}] {update['result']}")

ans2 = app2.get_state(cfg2).values
print(f"reviewer : {ans2['reviewer']}")

### Exercise 3 — Block at Manager Gate

**Expected behavior:** After rejecting Gate 1, the conditional edge routes to `END` immediately. The security gate node never runs. Final `result` = `'BLOCKED by manager'` and `security_approved = None`.

**What this illustrates:** Conditional edges after interrupt nodes let you short-circuit the rest of the pipeline based on human decisions — you don't need a separate rejection graph.

In [ ]:
# Answer — Exercise 3
cfg_ans3 = {"configurable": {"thread_id": "multi-ans3"}}

# Step 1: run to Gate 1
for chunk in multi_app.stream(
    {"action": "", "manager_approved": None, "security_approved": None, "result": ""},
    cfg_ans3,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        iv = chunk["__interrupt__"][0].value
        print(f"Paused at: {iv['gate']}")
        break

# Step 2: reject at manager gate
print("Rejecting at manager gate...")
for chunk in multi_app.stream(Command(resume=False), cfg_ans3, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"[{node}]: {update}")

ans3 = multi_app.get_state(cfg_ans3).values
print(f"\nresult            : {ans3['result']}")
print(f"manager_approved  : {ans3['manager_approved']}")
print(f"security_approved : {ans3['security_approved']}  <- never reached")